# ☕ Starbucks Beverage Nutrition Facts: Data Cleaning & Machine Learning Guide
### Based on the Official Class Colab Guide Workflow
---
This Google Colab notebook implements the complete 25-step data cleaning, exploratory data analysis (EDA), label encoding, feature splitting, Decision Tree, and Random Forest classification pipeline on the **Starbucks Beverage Nutrition Facts** dataset (`StarBucksNutritionFacts.csv.xlsx`).


In [ ]:
#Step 1. Import Libraries
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

from sklearn.preprocessing import LabelEncoder


In [ ]:
#Step 2. Upload Dataset
from google.colab import files
import os

# If dataset file is not in workspace, prompt upload
if not os.path.exists("StarBucksNutritionFacts.csv.xlsx"):
    print("📥 Upload your 'StarBucksNutritionFacts.csv.xlsx' file:")
    uploaded = files.upload()
else:
    print("✅ 'StarBucksNutritionFacts.csv.xlsx' already present in Colab session!")


In [ ]:
#Step 3. Load Dataset
try:
    df = pd.read_excel("StarBucksNutritionFacts.csv.xlsx")
except Exception:
    # Fallback if uploaded via files.upload()
    df = pd.read_excel(next(iter(uploaded)))

df.head()


In [ ]:
#Step 4. View Dataset Information
print("Dataset Dimensions (Shape):", df.shape)

df.info()


In [ ]:
#Step 5. Check Duplicate Records
duplicates = df.duplicated().sum()

print("Duplicate Records:", duplicates)


In [ ]:
#Step 6. Remove Duplicate Records
df = df.drop_duplicates()

print("Shape after drop_duplicates:", df.shape)


In [ ]:
#Step 7. Check Missing Values
print("Raw Missing Values Count:")
print(df.isnull().sum())


In [ ]:
#Step 8. Data Standardizing & Pre-Cleaning Specific Fields
# 1. Clean Column Headers (Remove leading/trailing spaces)
df.columns = df.columns.str.strip()

# 2. Trim hidden padding spaces in string columns
for col in df.select_dtypes(include=['object', 'string']).columns:
    df[col] = df[col].astype(str).str.strip()

# 3. Fix Typo in 'Total Fat (g)' ('3 2' -> '3.2') and convert to float
df['Total Fat (g)'] = df['Total Fat (g)'].str.replace('3 2', '3.2').astype(float)

# 4. Coerce text 'Varies' / 'varies' in 'Caffeine (mg)' to numeric NaN
df['Caffeine (mg)'] = pd.to_numeric(df['Caffeine (mg)'], errors='coerce')

print("Column Names Cleaned & Numeric Coercion Completed!")


In [ ]:
#Step 9. Fill Missing Numerical Values Using Median
numeric_columns = df.select_dtypes(include=np.number).columns

for col in numeric_columns:
    df[col] = df[col].fillna(df[col].median())

print("Missing Values Count After Imputation:")
print(df.isnull().sum())


In [ ]:
#Step 10. Encode Target Column (Beverage Category)
encoder = LabelEncoder()

df["Beverage_category_encoded"] = encoder.fit_transform(df["Beverage_category"])

df.head()


In [ ]:
#Step 11. Save Cleaned Dataset
df.to_csv("Cleaned_Dataset.csv", index=False)
print("✅ Cleaned_Dataset.csv saved successfully!")


In [ ]:
#STEP 12. Exploratory Data Analysis (EDA)
#Dataset Summary
df.describe()


In [ ]:
#Distribution of Beverage Category (Target)
plt.figure(figsize=(10,5))
sns.countplot(y='Beverage_category', data=df, palette='viridis', order=df['Beverage_category'].value_counts().index)
plt.title("Beverage Category Distribution")
plt.xlabel("Count")
plt.ylabel("Category")
plt.show()


In [ ]:
#Correlation Heatmap
plt.figure(figsize=(12,9))

numeric_df = df.select_dtypes(include=np.number)
sns.heatmap(numeric_df.corr(),
            annot=True,
            fmt='.2f',
            cmap="coolwarm")

plt.title("Correlation Matrix")

plt.show()


In [ ]:
#Calories Distribution
plt.figure(figsize=(7,4))

plt.hist(df["Calories"], bins=20, color='#00704A', edgecolor='black')

plt.xlabel("Calories (kcal)")
plt.ylabel("Frequency")
plt.title("Calories Distribution")

plt.show()


In [ ]:
#Sugars Distribution
plt.figure(figsize=(7,4))

plt.hist(df["Sugars (g)"], bins=20, color='#d9534f', edgecolor='black')

plt.xlabel("Sugars (g)")
plt.ylabel("Frequency")
plt.title("Sugars Distribution")

plt.show()


In [ ]:
#Caffeine Distribution
plt.figure(figsize=(7,4))

plt.hist(df["Caffeine (mg)"], bins=20, color='#f0ad4e', edgecolor='black')

plt.xlabel("Caffeine (mg)")
plt.ylabel("Frequency")
plt.title("Caffeine Distribution")

plt.show()


In [ ]:
#Step 13. Check all column names and encode remaining object features
print("All Columns:", df.columns.tolist())

le = LabelEncoder()

# Encode every column with object (text) datatype
for column in df.select_dtypes(include=['object']).columns:
    df[column] = le.fit_transform(df[column].astype(str))

df.head()


In [ ]:
#Step 14. Split Features and Target
# Target is Beverage_category_encoded
X = df.drop(columns=['Beverage_category', 'Beverage_category_encoded'])
y = df['Beverage_category_encoded']

print("Features Shape:", X.shape)
print("Target Shape:", y.shape)


In [ ]:
#Step 15. Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Train Features Shape:", X_train.shape)
print("Test Features Shape:", X_test.shape)


In [ ]:
#Step 16. Decision Tree
dt = DecisionTreeClassifier(random_state=42)

dt.fit(X_train, y_train)


In [ ]:
#Step 17. Decision Tree Prediction
dt_prediction = dt.predict(X_test)


In [ ]:
#Step 18. Decision Tree Accuracy
dt_accuracy = accuracy_score(y_test, dt_prediction)

print("Decision Tree Accuracy:", dt_accuracy)


In [ ]:
#Step 19. Decision Tree Report
print(classification_report(y_test, dt_prediction, target_names=encoder.classes_))


In [ ]:
#Step 20. Decision Tree Confusion Matrix
cm = confusion_matrix(y_test, dt_prediction)

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=encoder.classes_, yticklabels=encoder.classes_)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Decision Tree Confusion Matrix")
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.show()


In [ ]:
#Step 21. Random Forest
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train, y_train)


In [ ]:
#Step 22. Random Forest Prediction
rf_prediction = rf.predict(X_test)


In [ ]:
#Step 23. Random Forest Accuracy
rf_accuracy = accuracy_score(y_test, rf_prediction)

print("Random Forest Accuracy:", rf_accuracy)


In [ ]:
#Step 24. Random Forest Report
print(classification_report(y_test, rf_prediction, target_names=encoder.classes_))


In [ ]:
#Step 25. Random Forest Confusion Matrix
cm = confusion_matrix(y_test, rf_prediction)

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=encoder.classes_, yticklabels=encoder.classes_)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Random Forest Confusion Matrix")
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.show()
